# Experiment with local model code

Two independent examples: **feature selection** and **inference post-processing**.
Run the setup cell, then the example you need. Model functions run in this notebook's
Python process, using the packages installed in its kernel. No SageMaker job is started.
Use the same model dependencies as the approved runtime image.

The pattern is **prepare paths → set environment → import → call**. All outputs stay
under `.runs/model-experiments`. You can use another writable location, including
`/opt/ml`, by changing the example's root directory.

## Setup

- Locate this checkout and add its model source to Python's import path.
- Load standard notebook tools; all input preparation appears in the cells below.
- Keep model imports for later, after their environment variables are set.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import importlib
import json
import os
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "mlops-models/src/mlops_models").is_dir())
sys.path[:0] = [str(ROOT / "mlops-models/src"), str(ROOT / "mlops-models")]

print("Local checkout:", ROOT)
print("Notebook Python:", sys.executable)

## 1. Feature selection: prepare the three paths

`data_directory` contains `data.csv`. `output_directory` receives the selection
outputs. `parameter_path` points to the selected model's existing JSON file.

- Choose the model folder and create a fresh local input/output directory.
- Read `csv_input` from `notebooks/config.json`, accepting a local file or S3 object.
- Copy that CSV unchanged to `data_directory/data.csv`; keep the parameter file unchanged.
- Print all three paths so the inputs and output are visible before calling the model.

In [ ]:
import shutil
import boto3

MODEL_ID = "usecase1"  # Choose your actual model folder.
feature_root = ROOT / ".runs/model-experiments" / MODEL_ID / datetime.now(timezone.utc).strftime("selection-%Y%m%dT%H%M%S%f")
data_directory = feature_root / "input"
output_directory = feature_root / "output"
parameter_path = ROOT / "mlops-models/src/mlops_models/models" / MODEL_ID / "parameters.json"

config_path = ROOT / "notebooks/config.json"
csv_input = json.loads(config_path.read_text()).get("csv_input", "").strip()
if not csv_input:
    raise ValueError(f"Set csv_input in {config_path} to your local CSV or S3 object.")
if not parameter_path.is_file():
    raise FileNotFoundError(parameter_path)
data_directory.mkdir(parents=True)
output_directory.mkdir()
local_csv = data_directory / "data.csv"
if csv_input.startswith("s3://"):
    bucket, key = csv_input.removeprefix("s3://").split("/", 1)
    with boto3.client("s3").get_object(Bucket=bucket, Key=key)["Body"] as source, local_csv.open("wb") as destination:
        shutil.copyfileobj(source, destination)
else:
    source = Path(csv_input).expanduser()
    if not source.is_absolute():
        source = ROOT / source
    shutil.copy2(source, local_csv)

print("data_directory:", data_directory)
print("output_directory:", output_directory)
print("parameter_path:", parameter_path)

### Set the environment and read the inputs
- Set the model and output paths before importing feature-selection code.
- Disable cloud tracking and run-report writes for this local experiment.
- Load the existing JSON parameters and read `data_directory/data.csv` using the model's reader.
- Display a few rows; the selector below receives the full dataframe.

In [ ]:
os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_DIR": str(parameter_path.parent),
    "SM_MODEL_DIR": str(parameter_path.parent),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_TRAINING_MODE": "1",
    "MLOPS_MODEL_METADATA_URI": "none",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
parameters = json.loads(parameter_path.read_text())
from mlops_models.model_base.training import feature_selection
feature_selection = importlib.reload(feature_selection)
print("Feature-selection code:", feature_selection.__file__)
df = feature_selection.get_container_input(data_directory, parameters)
display(df.head())

### Fit and inspect the selector
- Construct the selector with the parameters loaded from `parameter_path`.
- Fit it on the dataframe returned by `get_container_input`.
- Export the selected features to `output_directory/selection` and display the result.

These are the same steps performed by `run_feature_selection(data_directory,
output_directory, parameters, selector_cls=feature_selection.FeatureSelector)`.
They are shown separately so you can inspect or change the dataframe and selector.
After editing the module, rerun the previous import cell and this cell.

In [ ]:
import pandas as pd

selector = feature_selection.FeatureSelector(parameters)
selector.fit(df)
selection_directory = feature_selection.write_container_output(selector, output_directory)
display(pd.read_csv(selection_directory / "selected_features.csv"))
print("Selection output:", selection_directory)

## 2. Inference post-processing: select an execution

Run this section after **Setup**; it does not need the feature-selection section.
Paste one inference execution ARN. An execution ID also works if notebook 02's
publication is saved for the configured namespace.

The following cells read the execution's actual **Postprocess Processing job**
and download its inputs. A failed job works too, provided its inputs still exist.
The model archive supplies input files; the code you run comes from this checkout.

- Select the inference execution whose post-processing you want to reproduce.
- Resolve a bare execution ID using the saved inference pipeline ARN, if needed.
- Create SageMaker and S3 clients in that execution's region for read-only calls.

In [ ]:
import boto3
from botocore.config import Config
import re

INFERENCE_EXECUTION = ""  # Paste the full inference execution ARN, or its execution ID.
postprocess_root = ROOT / ".runs/model-experiments" / datetime.now(timezone.utc).strftime("postprocess-%Y%m%dT%H%M%S%f")

execution_arn = INFERENCE_EXECUTION.strip()
if not execution_arn:
    raise ValueError("Set INFERENCE_EXECUTION to an inference execution ARN or ID.")
if not execution_arn.startswith("arn:"):
    config_path = ROOT / "notebooks/config.json"
    config = json.loads(config_path.read_text()) if config_path.is_file() else {}
    publication_path = ROOT / ".runs/source-deploy" / config.get("namespace", "") / "publication.json"
    publication = json.loads(publication_path.read_text()) if publication_path.is_file() else {}
    pipeline_arn = publication.get("pipelines", {}).get("inference", {}).get("arn", "")
    if not re.fullmatch(r"[A-Za-z0-9-]+", execution_arn) or not pipeline_arn:
        raise ValueError("For an execution ID, run notebook 02 first or supply the full execution ARN.")
    execution_arn = pipeline_arn.rstrip("/") + "/execution/" + execution_arn
if not re.fullmatch(r"arn:aws[a-z-]*:sagemaker:[a-z0-9-]+:\d{12}:pipeline/[^/]+/execution/[^/]+", execution_arn):
    raise ValueError("Expected a SageMaker pipeline execution ARN.")

region = execution_arn.split(":")[3]
session = boto3.Session(region_name=region)
limits = Config(connect_timeout=10, read_timeout=30,
                retries={"mode": "standard", "total_max_attempts": 3})
sm = session.client("sagemaker", config=limits)
s3 = session.client("s3", config=limits)
print("Execution:", execution_arn)

### Find the actual Postprocess job

- List the selected execution's steps and find its Postprocess step.
- Follow a cached step to its producing execution when it has no job ARN of its own.
- Describe that Processing job and print its identity and status, including a failed job.

In [ ]:
source_execution = execution_arn
seen_executions = set()
while source_execution not in seen_executions:
    seen_executions.add(source_execution)
    steps = [
        step
        for page in sm.get_paginator("list_pipeline_execution_steps").paginate(
            PipelineExecutionArn=source_execution)
        for step in page["PipelineExecutionSteps"]
        if step["StepName"] == "Postprocess"
    ]
    if len(steps) != 1:
        raise ValueError(f"{source_execution}: no single Postprocess step. Choose a Processing inference execution.")
    step = steps[0]
    job_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if job_arn:
        job = sm.describe_processing_job(ProcessingJobName=job_arn.rsplit("/", 1)[-1])
        break
    cached_execution = step.get("CacheHitResult", {}).get("SourcePipelineExecutionArn")
    if not cached_execution:
        raise ValueError(f"Postprocess has no job yet (status: {step['StepStatus']}).")
    print("Reading cached Postprocess inputs from:", cached_execution)
    source_execution = cached_execution
else:
    raise ValueError("Postprocess cache references form a cycle.")

print("Postprocess job:", job["ProcessingJobArn"])
print("Job status:", job["ProcessingJobStatus"])

### Map the job's inputs to local paths

`data_directory` contains `data.csv`; `selection_file` names a CSV file.
`prediction_directory` holds the score files, and `model_directory` holds the
unpacked model. `output_directory` receives the local analysis results.
Change `postprocess_root` above to use another writable location, including `/opt/ml`.

- Read the model ID and analysis mode from the job's arguments and environment.
- Assign the five paths passed to the local post-processing function.
- Match the four input directories to this job's S3 mounts and print their locations.

In [ ]:
import argparse

environment = job.get("Environment", {})
parser = argparse.ArgumentParser(add_help=False)
parser.add_argument("--model", default=environment.get("MODEL_ID", ""))
parser.add_argument("--analysis-mode", default=environment.get("MLOPS_ANALYSIS_MODE", "prediction_inference"))
args, _ = parser.parse_known_args(job.get("AppSpecification", {}).get("ContainerArguments", []))
model_id = args.model
analysis_mode = args.analysis_mode
if not model_id or analysis_mode not in {"prediction_test", "prediction_inference"}:
    raise ValueError("The job must identify the model and a supported analysis mode.")
if environment.get("MLOPS_PREDICTION_FORMAT", "csv") != "csv":
    raise ValueError("This local example expects the Processing CSV prediction route.")

postprocess_root = postprocess_root.expanduser().resolve()
data_directory = postprocess_root / "raw"
selection_file = postprocess_root / "selection/selected_features.csv"
prediction_directory = postprocess_root / "input"
model_directory = postprocess_root / "model"
output_directory = postprocess_root / "output"

mounts = {
    item["S3Input"]["LocalPath"].rstrip("/"): item["S3Input"]["S3Uri"]
    for item in job.get("ProcessingInputs", []) if "S3Input" in item
}
input_directories = {
    "/opt/ml/processing/raw": data_directory,
    "/opt/ml/processing/selection": selection_file.parent,
    "/opt/ml/processing/input": prediction_directory,
    "/opt/ml/processing/model": model_directory,
}
for mount, destination in input_directories.items():
    if mount not in mounts:
        raise ValueError(f"Postprocess input {mount} is missing; available mounts: {sorted(mounts)}")
    print(f"{mounts[mount]} -> {destination}")
print("Model:", model_id, "| Analysis mode:", analysis_mode)

### Download the four inputs

- Create a fresh local directory so files from different executions cannot be mixed.
- Download each exact S3 object or the files below its S3 folder prefix.
- Preserve filenames and nested folders, and print each completed local file.

In [ ]:
from pathlib import PurePosixPath
from botocore.exceptions import ClientError
import shutil

postprocess_root.mkdir(parents=True, exist_ok=False)
for mount, destination in input_directories.items():
    uri = mounts[mount]
    if not uri.startswith("s3://") or "/" not in uri[5:]:
        raise ValueError(f"Expected an S3 input location: {uri}")
    bucket, key = uri[5:].split("/", 1)
    if not key:
        raise ValueError("An input must name an object or folder, not the bucket root.")
    destination.mkdir()
    try:
        s3.head_object(Bucket=bucket, Key=key)
    except ClientError as error:
        if error.response["Error"]["Code"] not in {"404", "NoSuchKey", "NotFound"}:
            raise
        prefix = key.rstrip("/") + "/"
        objects = [
            (item["Key"], item["Key"][len(prefix):])
            for page in s3.get_paginator("list_objects_v2").paginate(Bucket=bucket, Prefix=prefix)
            for item in page.get("Contents", [])
            if not item["Key"].endswith("/")
        ]
    else:
        objects = [(key, key.rsplit("/", 1)[-1])]
    if not objects:
        raise FileNotFoundError(f"No input files found at {uri}")
    for object_key, relative in objects:
        name = PurePosixPath(relative)
        if name.is_absolute() or ".." in name.parts or not name.name:
            raise ValueError(f"Unsafe S3 filename: {relative}")
        target = destination / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        with s3.get_object(Bucket=bucket, Key=object_key)["Body"] as source, target.open("xb") as output:
            shutil.copyfileobj(source, output)
        print("Downloaded:", target, flush=True)

### Unpack the model and check the files

- Extract the downloaded model archive if its parameters are not already unpacked.
- Check that data, selected features, predictions and both model JSON files exist.
- Verify the metadata checksum when the job recorded one, then create the output directory.

In [ ]:
import hashlib
import tarfile

if not (model_directory / "parameters.json").is_file():
    archives = list(model_directory.glob("*.tar.gz"))
    if len(archives) != 1:
        raise ValueError(f"Expected one model archive in {model_directory}; found {len(archives)}.")
    with tarfile.open(archives[0]) as archive:
        archive.extractall(model_directory, filter="data")

for path in (data_directory / "data.csv", selection_file,
             model_directory / "parameters.json", model_directory / "model_metadata.json"):
    if not path.is_file():
        raise FileNotFoundError(f"Downloaded inputs are missing {path}")
expected_checksum = environment.get("MLOPS_MODEL_METADATA_SHA256")
if expected_checksum:
    actual_checksum = hashlib.sha256((model_directory / "model_metadata.json").read_bytes()).hexdigest()
    if actual_checksum != expected_checksum:
        raise ValueError("The downloaded model metadata does not match the checksum recorded by this job.")
if not any(p.is_file() and p.suffix.lower() == ".csv" for p in prediction_directory.rglob("*")):
    raise FileNotFoundError("The downloaded predictions contain no CSV files.")
output_directory.mkdir(exist_ok=True)
print("Local inputs ready:", postprocess_root)

### Set the environment before importing model code

- Point the model-directory variables at the unpacked parameters and metadata.
- Set the output directory, model ID and analysis mode used by this local run.
- Disable tracking/report uploads and print the arguments for the function call below.

In [ ]:
os.environ.update({
    "MODEL_ID": model_id,
    "MLOPS_MODEL_DIR": str(model_directory),
    "SM_MODEL_DIR": str(model_directory),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
    "MLOPS_ANALYSIS_MODE": analysis_mode,
    "MLOPS_PREDICTION_FORMAT": "csv",
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})
print("data_directory:", data_directory)
print("selection_file:", selection_file)
print("prediction_directory:", prediction_directory)
print("output_directory:", output_directory)
print("model_directory:", model_directory)
print("analysis_mode:", analysis_mode)

### Run or debug the local post-processing code

- Import the local base module after setting its environment variables.
- Reload it so rerunning this cell picks up edits to that module.
- Call `postprocess_predictions` directly with the prepared paths and analysis mode.

Set breakpoints inside that function and use **Debug Cell** in VS Code to step into
it. After editing the module, rerun this cell using the same downloaded inputs.
Restart the kernel after switching models or editing dependencies loaded by another module.

In [ ]:
from mlops_models.model_base.inference import postprocess
postprocess = importlib.reload(postprocess)
print("Post-processing code:", postprocess.__file__)
postprocess.postprocess_predictions(
    data_directory=data_directory,
    selection_file=selection_file,
    prediction_directory=prediction_directory,
    output_directory=output_directory,
    model_directory=model_directory,
    analysis_mode=analysis_mode,
)

### Inspect the outputs
- List the files written by your post-processing code.
- Display the example analysis CSV when that file exists.
- Keep the prepared inputs for another local edit/run cycle; no new SageMaker execution is needed.

In [ ]:
for output_file in sorted(output_directory.rglob("*")):
    if output_file.is_file():
        print(output_file)
analysis_csv = output_directory / "prediction_analysis.csv"
if analysis_csv.is_file():
    import pandas as pd
    saved_parameters = json.loads((model_directory / "parameters.json").read_text())
    display(pd.read_csv(analysis_csv, dtype={saved_parameters["primary_key"]: str}).head())